# Capa Oro — TFM TUI (Colab)

**Objetivo:** tabla(s) finales que alimentan el dashboard.

Qué se hace aquí:
- **Cruce espacial** (point-in-polygon): cada POI / restaurante se asigna a su barrio
  según dónde cae realmente, no por texto (más robusto).
- **Accesibilidad por punto**: número de paradas de transporte en un radio de **400 m**
  (~5 min andando). El cálculo se hace en UTM (EPSG:25830) para que sean metros reales.
- **Agregado por barrio**: densidad de oferta, % terrazas, diversidad de POI y accesibilidad media.

Salidas (carpeta `Oro/`):
- `POI.parquet` / `Restaurantes.parquet` — tablas de puntos con barrio y `n_paradas_400m` (marcadores del mapa).
- `Barrios.parquet` — una fila por barrio con los KPIs (gráficos y tablas del dashboard).
- `Barrios.geojson` — lo mismo pero con la geometría de los polígonos, en lat/lon (coropleto del mapa).

## 1. Setup — montaje, rutas y funciones base

In [7]:
!pip install -q geopandas

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import geopandas as gpd

BASE  = "/content/drive/MyDrive/Master/TFM TUI 3"
PLATA = f"{BASE}/Plata"
RAW   = f"{BASE}/Raw"
ORO   = f"{BASE}/Oro"
os.makedirs(ORO, exist_ok=True)

UTM   = 25830   # ETRS89 30N -> distancias en metros
RADIO = 400     # metros para contar paradas cercanas


def a_geo(df):
    """DataFrame con lat/lon -> GeoDataFrame en UTM (metros)."""
    g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326)
    return g.to_crs(UTM)


def guardar(df, tabla):
    """Guarda como Parquet plano (sin geometria) en la carpeta Oro."""
    if "geometry" in df.columns:
        df = df.drop(columns="geometry")
    df.to_parquet(f"{ORO}/{tabla}.parquet", index=False)
    print(f"OK  {tabla}  ->  {len(df)} filas, {df.shape[1]} columnas")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Cargar Plata + polígonos de barrios

Los barrios vienen en UTM (EPSG:25830). Nos quedamos con el nombre del barrio,
el distrito y el área (ya calculada en el shapefile, en m²).

In [8]:
poi  = a_geo(pd.read_parquet(f"{PLATA}/POI.parquet"))
rest = a_geo(pd.read_parquet(f"{PLATA}/Restaurantes.parquet"))
par  = a_geo(pd.read_parquet(f"{PLATA}/Paradas.parquet"))

barrios = gpd.read_file(f"{RAW}/CM-Poligono Barrios").to_crs(UTM)
barrios = (barrios[["NOMBRE", "NOMDIS", "Area", "geometry"]]
           .rename(columns={"NOMBRE": "barrio", "NOMDIS": "distrito", "Area": "area_m2"}))

print(f"Barrios: {len(barrios)} | POI: {len(poi)} | Rest: {len(rest)} | Paradas: {len(par)}")

Barrios: 131 | POI: 1393 | Rest: 98344 | Paradas: 9553


## 3. Enriquecer puntos — barrio (cruce espacial) + paradas en 400 m

El barrio de cada punto se decide por **dónde cae** dentro de los polígonos
(`within`), no por el campo de texto de Plata. Se descartan los `barrio`/`distrito`
originales para que la única fuente sea el polígono.

In [9]:
def contar_paradas(puntos, paradas, radio=RADIO):
    """Cuenta paradas dentro de <radio> metros de cada punto (usa indice espacial)."""
    buff = puntos[["geometry"]].copy()
    buff["geometry"] = buff.geometry.buffer(radio)
    j = gpd.sjoin(buff, paradas[["geometry"]], predicate="contains", how="inner")
    conteo = j.groupby(j.index).size()
    return conteo.reindex(puntos.index, fill_value=0)


def enriquecer(puntos, barrios, paradas):
    puntos = puntos.drop(columns=[c for c in ["barrio", "distrito"] if c in puntos.columns])
    p = gpd.sjoin(puntos, barrios[["barrio", "distrito", "geometry"]],
                  predicate="within", how="left").drop(columns="index_right")
    p["n_paradas_400m"] = contar_paradas(puntos, paradas)
    return p


poi_oro  = enriquecer(poi,  barrios, par)
rest_oro = enriquecer(rest, barrios, par)

print("POI sin barrio asignado: ", poi_oro["barrio"].isna().sum())
print("Rest sin barrio asignado:", rest_oro["barrio"].isna().sum())
display(rest_oro[["nombre", "barrio", "distrito", "tiene_terraza", "n_paradas_400m"]].head())

POI sin barrio asignado:  0
Rest sin barrio asignado: 3


,nombre,barrio,distrito,tiene_terraza,n_paradas_400m
0,VITACA,Justicia,Centro,True,13
1,ZAATAR & CO,Universidad,Centro,True,5
2,HOTEL MEDIODIA,Embajadores,Centro,False,17
3,MUNE,Justicia,Centro,False,14
4,LA DESCUBIERTA,Sol,Centro,False,13


## 4. Agregar por barrio

Una fila por barrio (los 131) con los KPIs. Ahora los KPIs de restaurantes se
calculan **solo sobre `macro_categoria == "restaurantes"`** (antes contaban todos
los negocios del censo). Se añaden:

- **`n_negocios`** y **`n_<macro>`**: locales por barrio, total y por grupo.
- **`pct_<macro>`**: densidad de cada grupo sobre el total del barrio (KPI 1).
- **`pct_turistico`**: % de oferta turística (restaurantes+cultura+ocio+hoteles).
- **`diversidad`**: índice de mezcla 0-1 (0 = un solo tipo domina, 1 = repartido).
- **`accesibilidad_rest`** y **`accesibilidad_media`**: paradas 400 m, en restaurantes y en todos los negocios.
- **`pct_terrazas`**: % de **restaurantes** con terraza (ya no diluido por otros negocios).

In [10]:
import numpy as np

# ---- POI: conteo y diversidad de tipos de POI ----
poi_agg = (poi_oro.groupby("barrio")
           .agg(n_poi=("nombre", "size"),
                diversidad_poi=("categoria", "nunique"))
           .reset_index())

# ---- Negocios (censo): total por barrio ----
neg_total = rest_oro.groupby("barrio").size().rename("n_negocios")

# ---- Conteo por macro-categoria: una columna n_<macro> por grupo ----
macro_counts = (rest_oro.groupby(["barrio", "macro_categoria"]).size()
                .unstack(fill_value=0))
macro_counts.columns = [f"n_{c}" for c in macro_counts.columns]

# ---- Densidad: % de cada macro sobre el total del barrio (KPI 1) ----
macro_pct = macro_counts.div(neg_total, axis=0).mul(100).round(1)
macro_pct.columns = [c.replace("n_", "pct_") for c in macro_counts.columns]

# ---- % turistico = restaurantes + cultura + ocio + hoteles ----
cols_tur = [f"n_{c}" for c in ["restaurantes", "cultura", "ocio", "hoteles"]
            if f"n_{c}" in macro_counts.columns]
pct_turistico = (macro_counts[cols_tur].sum(axis=1) / neg_total * 100).round(1).rename("pct_turistico")

# ---- Diversidad (mezcla) normalizada 0-1: 0 = un tipo domina, 1 = repartido ----
def diversidad_mezcla(fila):
    c = fila.values.astype(float); s = c.sum()
    if s == 0: return 0.0
    p = c[c > 0] / s; k = len(p)
    if k <= 1: return 0.0
    return round(float(-(p * np.log(p)).sum() / np.log(k)), 3)   # Shannon / ln(k) = evenness
diversidad = macro_counts.apply(diversidad_mezcla, axis=1).rename("diversidad")

# ---- KPIs SOLO de restaurantes (macro_categoria == "restaurantes") ----
rest_solo = rest_oro[rest_oro["macro_categoria"] == "restaurantes"]
pct_terrazas       = (rest_solo.groupby("barrio")["tiene_terraza"].mean() * 100).round(1).rename("pct_terrazas")
accesibilidad_rest = rest_solo.groupby("barrio")["n_paradas_400m"].mean().round(1).rename("accesibilidad_rest")

# ---- Accesibilidad media global (todos los negocios) ----
accesibilidad_media = rest_oro.groupby("barrio")["n_paradas_400m"].mean().round(1).rename("accesibilidad_media")

# ---- Tabla final: 131 barrios + todos los KPIs (left join, no perder barrios sin oferta) ----
barrios_oro = barrios.drop(columns="geometry").copy()
barrios_oro["area_km2"] = (barrios_oro["area_m2"] / 1e6).round(3)
barrios_oro = barrios_oro.drop(columns="area_m2")

# n_restaurantes headline (desde el conteo por macro)
n_restaurantes = macro_counts["n_restaurantes"].rename("n_restaurantes") if "n_restaurantes" in macro_counts.columns else None

partes = [poi_agg.set_index("barrio"), neg_total, n_restaurantes, macro_pct,
          pct_turistico, diversidad, pct_terrazas, accesibilidad_rest, accesibilidad_media]
for parte in partes:
    if parte is not None:
        barrios_oro = barrios_oro.merge(parte, on="barrio", how="left")

# Rellenar barrios sin oferta y tipar los conteos
for c in ["n_poi", "diversidad_poi", "n_negocios", "n_restaurantes"]:
    if c in barrios_oro.columns:
        barrios_oro[c] = barrios_oro[c].fillna(0).astype(int)
barrios_oro = barrios_oro.fillna(0)

# Densidades por km2
barrios_oro["densidad_negocios"] = (barrios_oro["n_negocios"] / barrios_oro["area_km2"]).round(1)
barrios_oro["densidad_poi"]      = (barrios_oro["n_poi"]      / barrios_oro["area_km2"]).round(1)

display(barrios_oro.sort_values("densidad_negocios", ascending=False).head())

,barrio,distrito,area_km2,n_poi,diversidad_poi,n_negocios,n_restaurantes,pct_alimentacion,pct_comercio,pct_cultura,...,pct_restaurantes,pct_salud,pct_servicios,pct_turistico,diversidad,pct_terrazas,accesibilidad_rest,accesibilidad_media,densidad_negocios,densidad_poi
5,Sol,Centro,0.445,42,3,1282,431,11.9,33.5,0.7,...,33.6,1.7,7.0,39.1,0.727,41.8,19.3,19.7,2880.9,94.4
37,Gaztambide,Chamberí,0.506,6,3,1261,213,10.8,19.3,0.8,...,16.9,8.6,27.0,22.0,0.836,36.2,13.2,12.0,2492.1,11.9
39,Trafalgar,Chamberí,0.612,3,3,1485,285,8.6,15.4,1.8,...,19.2,7.3,29.1,25.7,0.843,42.8,13.0,12.5,2426.5,4.9
19,Recoletos,Salamanca,0.873,13,4,1910,252,3.9,34.1,1.3,...,13.2,4.5,28.2,17.1,0.737,35.3,8.8,9.0,2187.9,14.9
1,Embajadores,Centro,1.034,30,5,2219,422,12.5,31.1,2.7,...,19.0,2.2,15.9,26.9,0.813,30.8,7.0,6.7,2146.0,29.0


## 5. Guardar salidas

Puntos y KPIs en Parquet; barrios también en GeoJSON (con geometría, en lat/lon)
para el coropleto del mapa.

In [11]:
guardar(poi_oro,     "POI")
guardar(rest_oro,    "Restaurantes")
guardar(barrios_oro, "Barrios")

# GeoJSON con geometria (lat/lon) + KPIs -> para pintar los barrios en el mapa
barrios_geo = (barrios[["barrio", "geometry"]].to_crs(4326)
               .merge(barrios_oro, on="barrio", how="left"))
barrios_geo.to_file(f"{ORO}/Barrios.geojson", driver="GeoJSON")
print("OK  Barrios.geojson  (con geometria, lat/lon)")

OK  POI  ->  1393 filas, 8 columnas
OK  Restaurantes  ->  98344 filas, 12 columnas
OK  Barrios  ->  131 filas, 25 columnas
OK  Barrios.geojson  (con geometria, lat/lon)


## 6. Comprobación rápida

In [12]:
b = pd.read_parquet(f"{ORO}/Barrios.parquet")
r = pd.read_parquet(f"{ORO}/Restaurantes.parquet")

print("Top 5 barrios por densidad de negocios:")
print(b.nlargest(5, "densidad_negocios")[
    ["barrio", "distrito", "n_negocios", "n_restaurantes", "pct_turistico", "diversidad", "pct_terrazas"]
].to_string(index=False))

print("\nBarrios sin restaurantes:", int((b["n_restaurantes"] == 0).sum()))
print("Rango diversidad (0-1):", b["diversidad"].min(), "-", b["diversidad"].max())
print("\nDistribucion de n_paradas_400m (restaurantes):")
print(r[r["macro_categoria"] == "restaurantes"]["n_paradas_400m"].describe().round(1).to_string())

Top 5 barrios por densidad de negocios:
     barrio  distrito  n_negocios  n_restaurantes  pct_turistico  diversidad  pct_terrazas
        Sol    Centro        1282             431           39.1       0.727          41.8
 Gaztambide  Chamberí        1261             213           22.0       0.836          36.2
  Trafalgar  Chamberí        1485             285           25.7       0.843          42.8
  Recoletos Salamanca        1910             252           17.1       0.737          35.3
Embajadores    Centro        2219             422           26.9       0.813          30.8

Barrios sin restaurantes: 0
Rango diversidad (0-1): 0.727 - 0.94

Distribucion de n_paradas_400m (restaurantes):
count    15628.0
mean         6.6
std          5.7
min          0.0
25%          3.0
50%          5.0
75%          9.0
max         36.0
